In [0]:
from pyspark.sql import functions as F

base = "/Volumes/workspace/default/raw"

def read(pattern):
    return spark.read.option("header", True).option("sep", "\t").csv(f"{base}/{pattern}")

def clean_cols(df):
    for c in df.columns:
        df = df.withColumnRenamed(c, c.strip().replace(" ", "_").replace("-", "_"))
    return df

def money(df, cols):
    for c in cols:
        df = df.withColumn(c, F.regexp_replace(F.col(c), r"[$,]", "").cast("double"))
    return df

def to_int(df, cols):
    for c in cols:
        df = df.withColumn(c, F.col(c).cast("int"))
    return df

def to_date_col(df, col):
    # "Friday, August 25, 2017" -> drop the weekday -> "August 25, 2017" -> date
    no_weekday = F.regexp_extract(F.col(col), r"^[A-Za-z]+,\s*(.*)$", 1)
    return df.withColumn(col, F.to_date(no_weekday, "MMMM d, yyyy"))

# fact_sales
sales = clean_cols(read("Sales.csv"))
sales = money(sales, ["Unit_Price", "Sales", "Cost"])
sales = to_int(sales, ["ProductKey", "ResellerKey", "EmployeeKey", "SalesTerritoryKey", "Quantity"])
sales = to_date_col(sales, "OrderDate")
sales.write.mode("overwrite").saveAsTable("workspace.default.fact_sales")

# dim_product
product = clean_cols(read("Product*.csv"))
product = to_int(money(product, ["Standard_Cost"]), ["ProductKey"])
product.write.mode("overwrite").saveAsTable("workspace.default.dim_product")

# dim_sales_territory
region = to_int(clean_cols(read("Region.csv")), ["SalesTerritoryKey"])
region.write.mode("overwrite").saveAsTable("workspace.default.dim_sales_territory")

# optional extras
reseller = to_int(clean_cols(read("Reseller.csv")), ["ResellerKey"])
reseller.write.mode("overwrite").saveAsTable("workspace.default.dim_reseller")

targets = money(clean_cols(read("Targets.csv")), ["Target"])
targets = to_date_col(targets, "TargetMonth")
targets.write.mode("overwrite").saveAsTable("workspace.default.targets")

for t in ["fact_sales", "dim_product", "dim_sales_territory", "dim_reseller", "targets"]:
    print(t, spark.table(f"workspace.default.{t}").count())

### Sidebar notes
- **Workspace:** where I store and organize notebooks, folders and files.
- **Catalog:** where I browse tables, schemas and volumes.
- **Compute:** where clusters and serverless compute are managed.
- **Workflows (Jobs & Pipelines):** where I schedule and run jobs and pipelines.
- **Data Ingestion:** where I bring data in from files and other sources.

In [0]:
display(spark.sql("DESCRIBE fact_sales"))
display(spark.sql("DESCRIBE dim_product"))
display(spark.sql("DESCRIBE dim_sales_territory"))

In [0]:
%sql
SELECT COUNT(*) FROM fact_sales

In [0]:
%sql
SELECT COUNT(*) FROM dim_product

In [0]:
%sql
SELECT COUNT(*) FROM dim_sales_territory